# Appendix. The gene selection comparison study

Question: did any of four ways of choosing a gene list score higher than the others, and did a chosen list add to the clinical covariates?

What enters: the saved tables and settings of the study, computed on the earlier data version (851 development patients) by the archived notebook at commit f8c4a00; nothing is recomputed.

What leaves: nothing new. The supervisors chose elastic-net Cox as the selector afterwards (22 October 2026).

Result: rendered under the imports from the saved tables.

How to run: the nbconvert command of notebook 1 with this file; seconds.

In [ ]:
import hashlib
import json
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

HC = "Harrell's C (mean)"
RUNS = Path("../results/appendix_gene_selection_comparison/runs")
TABLES = Path("../results/appendix_gene_selection_comparison/tables")

In [ ]:
compared = pd.read_csv(RUNS / "what_is_compared.csv", index_col=0)
methods = list(compared.index[compared["role"] == "selection method"])
genes_only = pd.read_csv(TABLES / "method_comparison_genes_only.csv", index_col=0)
plus_clinical = pd.read_csv(TABLES / "method_comparison_genes_plus_clinical.csv", index_col=0)
against_best = pd.read_csv(TABLES / "method_comparison_against_best.csv", index_col=0)
tied = against_best[(against_best["interval low"] <= 0) & (against_best["interval high"] >= 0)]
best = genes_only.loc[methods, HC].idxmax()
display(Markdown(f"Result: the four methods are tied with genes only (highest mean C {best} at "
                 f"{genes_only.loc[best, HC]:.3f}, {len(tied)} others within its interval); no "
                 f"selected list added to the clinical covariates (clinical only "
                 f"{plus_clinical.loc['clinical only', HC]:.3f})."))

## What was compared

Two reference rows without genes and four selection methods, each tuned inside the inner folds of a nested cross-validation and scored by a ridge Cox model on the outer test parts, with genes only and with genes plus the clinical covariates.

In [ ]:
compared

In [ ]:
outcome_driven = compared.loc[methods, "uses patient outcomes to pick genes"] == "yes"
display(Markdown(f"{int(outcome_driven.sum())} of the {len(methods)} selection methods use the "
                 "outcome to pick genes; the references pick no genes."))

## Settings of the runs

The settings as saved beside the runs, one row per setting; the saved file also says what each one is.

In [ ]:
settings = json.loads((RUNS / "settings.json").read_text())
settings_table = pd.DataFrame(settings).T[["value"]]
settings_table

## Results of both scoring forms

Mean Harrell's C over the outer folds, its standard deviation, Uno's C and the median list size, with genes only and with genes plus the clinical covariates.

In [ ]:
display(genes_only.round(3), plus_clinical.round(3))

In [ ]:
display(Markdown(f"With genes only the highest mean among the methods is {best} at "
                 f"{genes_only.loc[best, HC]:.3f}; with the clinical covariates no "
                 f"method rises above clinical only at "
                 f"{plus_clinical.loc['clinical only', HC]:.3f}."))

## Paired comparisons on both metrics

Each row is a difference between two rows scored on the same outer folds, with its 95% interval from a corrected resampled t over 25 folds; an interval that includes 0 shows no difference.

In [ ]:
paired = pd.read_csv(TABLES / "paired_comparisons.csv", index_col=0)
paired

In [ ]:
both = pd.read_csv(TABLES / "both_metrics.csv", index_col=0)
includes = int(both["Harrell's C: interval includes 0"].sum())
display(Markdown(f"{includes} of the {len(both)} intervals on Harrell's C include 0, and "
                 f"{int(both[both.columns[-2]].sum())} of {len(both)} on Uno's C."))

## Leakage check

The same methods with the gene list chosen on all patients before the folds, against selection inside the training fold; the difference is the score that leaks in.

In [ ]:
leakage = pd.read_csv(TABLES / "leakage_check_table.csv", index_col=0)
leakage.round(3)

In [ ]:
display(Markdown(f"Selecting genes on all patients inflates Harrell's C by "
                 f"{leakage['difference'].min():+.3f} to {leakage['difference'].max():+.3f}."))

## The choice rule

The rule fixed before the runs: take the highest mean with genes only, count as tied every method whose interval against it includes 0, and among the tied prefer the method that does not use the outcome.

In [ ]:
uses_outcome = compared.loc[tied.index, "uses patient outcomes to pick genes"]
no_outcome = list(uses_outcome.index[uses_outcome == "no"])
display(Markdown(f"Highest mean with genes only: {best}. Tied with it: {', '.join(tied.index)}. "
                 f"The rule picks {', '.join(no_outcome)}. The supervisors chose elastic-net Cox "
                 "afterwards, which this study had found tied with the others."))

## Run record

The commit of this notebook, the package versions, and the commit and hashes recorded by the archived runs notebook.

In [ ]:
def git(*arguments):
    """The output of one git command, for the run record."""
    command = ["git", *arguments]
    return subprocess.run(command, capture_output=True, text=True, check=True).stdout.strip()


runs_record = json.loads((RUNS / "run_record.json").read_text())
tables_hash = hashlib.sha256()
for path in sorted(TABLES.glob("*.csv")):
    tables_hash.update(path.read_bytes())
run_record = {"git commit": git("rev-parse", "HEAD"),
              "tracked files changed since that commit":
                  git("status", "--porcelain", "--untracked-files=no").splitlines(),
              "python": sys.version.split()[0], "pandas": version("pandas"),
              "runs computed at commit": runs_record.get("git commit"),
              "saved tables (sha256 of all csv files)": tables_hash.hexdigest()}
print(json.dumps(run_record, indent=1))